# A Revolução da Visão Computacional

Visão computacional usando a biblioteca YOLO da Ultralytics.

O Ultralytics YOLO26 é um _framework_ de Inteligência Artificial versátil que suporta múltiplas tarefas de visão computacional.
O _framework_ pode ser utilizado para realizar detecção, segmentação, caixas delimitadoras orientadas (OBB), classificação e estimativa de pose.
Cada uma dessas tarefas possui um objetivo e um caso de uso diferentes, permitindo que você resolva diversos desafios de visão computacional com um único _framework_.

**Não há mágica, há matemática!**

In [3]:
import sys
if "google.colab" not in sys.modules:
    raise RuntimeError("Este notebook deve ser executado no Google Colab.")


In [4]:
# instalar a biblioteca
!pip install -q -U ultralytics

# fazer o upload de uma imagem sugerida ou uma imagem sua
#!gdown "https://drive.google.com/u/1/link_da_imagem.jpeg"
#!wget "https://link_da_imagem.jpeg"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.7 MB/s eta 0:00:00


In [5]:
# carregar o modelo YOLOv26 pré-treinado e realizar a detecção de objetos na imagem carregada
#!yolo task=detect mode=predict model=yolo26n.pt conf=0.5 source='/content/imagem.jpeg'

# o resultado da detecção de objetos será salvo no diretório "/runs/detect/predictX"

In [6]:
# Seguindo os docs do YOLO
from ultralytics import YOLO

MODEL_PT = "yolo26n.pt"
EPOCHS = 10
IMGSZ = 640
BATCH = 16
FRACTION = 0.5  # usar apenas metade do dataset em cada epoch (treino mais rapido)
VIDEO_URL = "https://www.youtube.com/watch?v=2cEJ5R6T_Dg"
NDJSON = "objects-phone-v3.ndjson"  # arquivo enviado ao colab

# EMBED CODE
# <iframe src="https://platform.ultralytics.com/embed/edwin-campos-2/datasets/objects-phone-v3" width="480" height="320" frameborder="0" style="border-radius:12px"></iframe>

# LINK (link do dataset no Ultralytics Platform)
LINK = "https://platform.ultralytics.com/edwin-campos-2/datasets/objects-phone-v3"

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [7]:
# Montar o Google Drive (robusto ao erro "credentials-propagation 404" do Colab).
DRIVE = "/content/drive"

import os
from google.colab import drive

def drive_ja_montado():
    return os.path.isdir(os.path.join(DRIVE, "MyDrive"))

if drive_ja_montado():
    print("Google Drive ja montado em", DRIVE)
else:
    try:
        drive.mount(DRIVE)
    except Exception as e:
        print("drive.mount() falhou:", type(e).__name__, "-", e)
        print("Tentando o fluxo classico (_mount) ...")
        try:
            drive._mount(DRIVE)
        except Exception as e2:
            print("_mount() tambem falhou:", type(e2).__name__, "-", e2)
            print()
            print("O erro 404 em /credentials-propagation e um problema da sessao do Colab.")
            print("Tente, nesta ordem:")
            print("  1. Runtime > Restart session e execute o notebook novamente.")
            print("  2. No painel de arquivos (icone de pasta) da esquerda, clique em")
            print("     Mount Drive - usa o fluxo do navegador e costuma funcionar.")
            print("  3. Ou use a celula FALLBACK abaixo (envio via API REST do Drive).")
            raise

if not drive_ja_montado():
    raise RuntimeError("Google Drive nao montado. Veja as dicas acima ou use a celula FALLBACK.")


Mounted at /content/drive


In [8]:
# Baixar o NDJSON mais novo do Ultralytics Platform via API (check_file).
# A API key NAO fica salva no notebook:
#   1) le da secret 'ULTRALYTICS_API_KEY' do Colab (icone de cadeado a esquerda);
#   2) se nao existir, pede no prompt (getpass) - use o prefixo 'ul_'.
import os
from getpass import getpass
from google.colab import userdata

key = os.environ.get("ULTRALYTICS_API_KEY")
if not key:
    try:
        key = userdata.get("ULTRALYTICS_API_KEY")
    except Exception:
        key = None
if not key:
        key = getpass("Coloque sua API key do Ultralytics Platform: ")

os.environ["ULTRALYTICS_API_KEY"] = key

from ultralytics.utils.checks import check_file  # check_file mora em utils.checks

DATASET_URI = "ul://baguette/datasets/objects-phone-v3"  # username baguette
NDJSON = check_file(DATASET_URI, download_dir="/content/ndjson")
print("NDJSON mais novo baixado ->", NDJSON)
print("Tamanho:", round(os.path.getsize(NDJSON) / 1e6, 2), "MB")


NDJSON mais novo baixado -> /content/ndjson/baguette/datasets/objects-phone-v3/objects-phone-v3.ndjson
Tamanho: 10.42 MB


In [9]:
# Copiar o NDJSON para uma pasta do Drive (so roda se o Drive estiver montado).
import shutil

DESTINO_DRIVE = "/content/drive/MyDrive/LIA/objects-phone-v3.ndjson"
os.makedirs(os.path.dirname(DESTINO_DRIVE), exist_ok=True)
shutil.copy(NDJSON, DESTINO_DRIVE)

if os.path.getsize(DESTINO_DRIVE) == os.path.getsize(NDJSON):
    print("NDJSON copiado para o Drive ->", DESTINO_DRIVE)
else:
    raise RuntimeError("Copia incompleta para o Drive.")


NDJSON copiado para o Drive -> /content/drive/MyDrive/LIA/objects-phone-v3.ndjson


In [10]:
# Converter o NDJSON para o formato YOLO.
# A função baixa todas as imagens das URLs assinadas, organiza em images/{train,val,test}
# com labels/ correspondentes (um .txt por imagem) e gera o arquivo data.yaml.
# Se o dataset YOLO ja existir (images/ + labels/ + data.yaml), pula a conversao.
from pathlib import Path
CONVERTED_DIR = "/content/converted_datasets"
yaml_path = Path(CONVERTED_DIR) / "data.yaml"
if yaml_path.exists() and (Path(CONVERTED_DIR) / "images" / "train").exists():
    print("Dataset YOLO ja convertido. Pulando conversao ->", yaml_path)
else:
    # Jupyter/Colab ja roda um event loop: aplicar nest_asyncio permite usar asyncio.run().
    import asyncio
    import subprocess, sys
    try:
        import nest_asyncio
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nest-asyncio"])
        import nest_asyncio
    nest_asyncio.apply()

    from ultralytics.data.converter import convert_ndjson_to_yolo

    yaml_path = asyncio.run(convert_ndjson_to_yolo(NDJSON, output_path=CONVERTED_DIR))
    print("\ndata.yaml gerado ->", yaml_path)


Converting objects-phone-v3.ndjson fraction=1.0 → /content/converted_datasets/objects-phone-v3-0307fefc using 31911 train, 1877 val, 1257 test images: 32% ━━━╸──────── 11257/35045 405.7it/s 26.3s<58.6s

KeyboardInterrupt: 

In [ ]:
# Conferir o conteúdo do data.yaml e a contagem de imagens por split.
from ultralytics.utils import YAML
import yaml as _yaml

config = YAML.load(yaml_path)
print("Conteúdo do data.yaml:")
print(_yaml.safe_dump(config, default_flow_style=False, sort_keys=False))

for split in ("train", "val", "test"):
    n = len(list(yaml_path.parent.glob(f"images/{split}/*")))
    print(f"{split}: {n} imagens")


Conteúdo do data.yaml:
task: detect
name: objects phone v3
description: Street-level surveillance video frames capturing people using cellular
  phones in urban environments, annotated with bounding boxes for object detection
  tasks.
bytes: 3345261051
url: https://platform.ultralytics.com/baguette/datasets/objects-phone-v3
version: latest
created_at: '2026-09-02T05:03:58.215Z'
updated_at: '2026-09-02T05:03:58.215Z'
names:
  0: celular
test: images/test
train: images/train
val: images/val
hash: 0307fefc
complete: true

train: 31911 imagens
val: 1877 imagens
test: 1257 imagens


In [ ]:
# Treinar com transfer learning OU pular para o best.pt ja existente.
# Pergunta no prompt: digite "s" para re-treinar; qualquer outra coisa usa o arquivo existente.
# Se optar por re-treinar, as celulas de validacao e teste (proximas) rodam no modelo novo.
import glob

from ultralytics import YOLO

best_pts = sorted(glob.glob("/content/runs/detect/*/weights/best.pt"))
resposta = input("Re-treinar o modelo? (s/N): ").strip().lower()
RETREINAR = resposta in ("s", "sim", "y", "yes")

if RETREINAR:
    model = YOLO(MODEL_PT)  # modelo base pre-treinado (ex.: yolo26n.pt)
    results = model.train(
        data=yaml_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        fraction=FRACTION,  # usa apenas metade do dataset por epoch
    )
    best_path = str(model.trainer.best)
    print("Re-treino concluido. Melhores pesos em:", best_path)
elif best_pts:
    best_path = best_pts[-1]
    print("Usando best.pt existente (sem re-treino) ->", best_path)
    print("Validacao e teste abaixo rodarao sobre esse arquivo.")
else:
    print("Nenhum best.pt encontrado; treinando do zero.")
    model = YOLO(MODEL_PT)
    results = model.train(
        data=yaml_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        fraction=FRACTION,
    )
    best_path = str(model.trainer.best)
    print("Treino concluido. Melhores pesos em:", best_path)


Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/converted_datasets/objects-phone-v3-0307fefc/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name

/usr/lib/python3.13/pathlib/_local.py:277: RuntimeWarning: coroutine 'convert_ndjson_to_yolo' was never awaited
  @property


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: MuSGD(lr=0.01, momentum=0.9) with parameter groups 114 weight(decay=0.0), 126 weight(decay=0.0005), 126 bias(decay=0.0)
Plotting labels to /content/runs/detect/train-2/labels.jpg... 
Using 31911 train, 1877 val images for fraction=1.0 at imgsz=640
Using 2 dataloader workers
Logging results to /content/runs/detect/train-2
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       1/30      2.62G      1.662      3.223    0.01145         11        640: 100% ━━━━━━━━━━━━ 1995/1995 2.9it/s 11:290.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 59/59 1.4s/it 1:200.4sss
                   all       1877       1649      0.765      0.695       0.78      0.496

      Epoch    GPU_mem   box_loss   cls_loss    l1_los

: 

In [ ]:
# Validar com os melhores pesos do treino no split de validacao.
import glob

best_path = sorted(glob.glob("/content/runs/detect/*/weights/best.pt"))[-1]
print("Usando pesos ->", best_path)

val_model = YOLO(best_path)
metrics = val_model.val(data=yaml_path, split="val", imgsz=IMGSZ, batch=BATCH)
print("Boas metricas, box P/mAP50-95 ->", metrics.box.map)


In [ ]:
# Testar no split de teste (dados nunca vistos no treino/validacao).
test_model = YOLO(best_path)
metrics_test = test_model.val(data=yaml_path, split="test", imgsz=IMGSZ, batch=BATCH)
print("Teste, box P/mAP50-95 ->", metrics_test.box.map)


In [ ]:
# Baixar video do YouTube e salvar o original no Google Drive.
import subprocess, sys, shutil, glob

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yt-dlp'])
print("Baixando video ->", VIDEO_URL)
subprocess.run([sys.executable, '-m', 'yt_dlp',
    "-o", "/content/video_input.%(ext)s",
    VIDEO_URL])

video_input = glob.glob("/content/video_input.*")[-1]
print("Video baixado ->", video_input)

# DRIVE ja montado na celula 5; salvar em MyDrive/LIA/
VIDEO_DRIVE = DRIVE + "/MyDrive/LIA/" + os.path.basename(video_input)
shutil.copy(video_input, VIDEO_DRIVE)
print("Original salvo no Drive ->", VIDEO_DRIVE)


In [ ]:
# Rodar a deteccao no video baixado (conf baixa para captar mais celulares).
import shutil, glob
from ultralytics import YOLO

video_model = YOLO(best_path)
print("Modelo:", best_path)

video_model.predict(
    source=video_input,
    conf=0.15,  # conf baixa para detectar mais celulares
    imgsz=IMGSZ,
    save=True,
    stream=True,  # generator para nao acumular frames na RAM
    name="predict_video",
)
print("Predicao salva em /content/runs/detect/predict_video/")

# copiar video com caixas de deteccao para o Drive
detected = sorted(glob.glob('/content/runs/detect/predict_video/video_input*'))[-1]
DETECTED_DRIVE = DRIVE + "/MyDrive/LIA/" + os.path.basename(detected)
shutil.copy(detected, DETECTED_DRIVE)
print("Video detectado salvo no Drive ->", DETECTED_DRIVE)


In [ ]:
# Salvar no Google Drive: conversao (data.yaml), treino, validacao e predicao do video.
import shutil

RESULTADOS = "/content/drive/MyDrive/LIA/objects-phone-v3"
os.makedirs(RESULTADOS, exist_ok=True)

def copiar_para_drive(origem, destino):
    if not os.path.exists(origem):
        print("Nao encontrado, pulando:", origem)
        return
    if os.path.exists(destino):
        shutil.rmtree(destino)
    shutil.copytree(origem, destino)
    print("Salvo:", destino)

# 1) conversao: data.yaml (a estrutura images/ + labels/ ja esta em /content/converted_datasets)
shutil.copy(yaml_path, os.path.join(RESULTADOS, "data.yaml"))
print("data.yaml salvo.")

# 2) treino
for d in sorted(glob.glob("/content/runs/detect/train*"))[-1:]:
    copiar_para_drive(d, os.path.join(RESULTADOS, os.path.basename(d)))

# 3) validacao
for d in sorted(glob.glob("/content/runs/detect/val*"))[-1:]:
    copiar_para_drive(d, os.path.join(RESULTADOS, os.path.basename(d)))

# 4) predicao do video
for d in sorted(glob.glob("/content/runs/detect/predict_video*"))[-1:]:
    copiar_para_drive(d, os.path.join(RESULTADOS, os.path.basename(d)))

print("\nTudo salvo em:", RESULTADOS)
